# Clean Data Pipeline Test

This notebook validates `src/data/clean_data.py` end-to-end.

Run cells from top to bottom.

In [1]:
from pathlib import Path
import sys
import importlib
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
project_root = cwd if cwd.name == 'xai-retail-replenishment' else cwd.parent
src_path = project_root / 'src'
if str(src_path) in sys.path:
    sys.path.remove(str(src_path))
sys.path.insert(0, str(src_path))

import data.load_data as load_data_module
import data.clean_data as clean_data_module
importlib.reload(load_data_module)
importlib.reload(clean_data_module)

load_rossmann = load_data_module.load_rossmann
handle_missing_values = clean_data_module.handle_missing_values
remove_outliers = clean_data_module.remove_outliers
generate_synthetic_inventory_fields = clean_data_module.generate_synthetic_inventory_fields
clean_pipeline = clean_data_module.clean_pipeline

print('Project root:', project_root)
print('Using clean_data module:', clean_data_module.__file__)

Project root: C:\Users\amrok\Desktop\Thesis\Project_XAI\xai-retail-replenishment
Using clean_data module: C:\Users\amrok\Desktop\Thesis\Project_XAI\xai-retail-replenishment\src\data\clean_data.py


In [2]:
rossmann_dir = project_root / 'data' / 'raw' / 'rossmann'
ross = load_rossmann(str(rossmann_dir))

train = ross['train'].copy()
store = ross['store'].copy()
df = train.merge(store, on='Store', how='left')

print('train shape:', train.shape)
print('store shape:', store.shape)
print('merged shape:', df.shape)
print('Columns sample:', df.columns[:12].tolist())

C:\Users\amrok\Desktop\Thesis\Project_XAI\xai-retail-replenishment\src\data\load_data.py:27: DtypeWarning: Columns (0: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, parse_dates=parse_dates)


train shape: (1017209, 9)
store shape: (1115, 10)
merged shape: (1017209, 18)
Columns sample: ['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment', 'CompetitionDistance']


In [3]:
# Test handle_missing_values on a controlled sample
sample = df.head(10000).copy()
sample.loc[sample.sample(frac=0.02, random_state=42).index, 'CompetitionDistance'] = np.nan
sample.loc[sample.sample(frac=0.01, random_state=7).index, 'Open'] = np.nan

before_missing = sample[['CompetitionDistance', 'Open']].isna().sum()
filled = handle_missing_values(sample)
after_missing = filled[['CompetitionDistance', 'Open']].isna().sum()

print('Missing before:')
print(before_missing)
print('\nMissing after:')
print(after_missing)

Missing before:
CompetitionDistance    226
Open                   100
dtype: int64

Missing after:
CompetitionDistance    0
Open                   0
dtype: int64


In [4]:
# Test outlier logic
tmp = filled.copy()
tmp.loc[tmp.index[:5], 'Sales'] = tmp['Sales'].max() * 20
out = remove_outliers(tmp, columns=['Sales', 'Customers'])

print('Rows before outlier removal:', len(tmp))
print('Rows after outlier removal :', len(out))
print('is_outlier in output:', 'is_outlier' in out.columns)

Rows before outlier removal: 10000
Rows after outlier removal : 9838
is_outlier in output: True


In [5]:
# Test synthetic inventory fields
enriched = generate_synthetic_inventory_fields(out, seed=42)
required = [
    'stock_on_hand',
    'lead_time_days',
    'reorder_cost',
    'supplier_min_order_qty',
    'supplier_order_multiple',
]

print('Required synthetic columns present:')
print({c: (c in enriched.columns) for c in required})
print('\nQuick stats:')
print(enriched[required].describe().T[['mean', 'min', 'max']])

Required synthetic columns present:
{'stock_on_hand': True, 'lead_time_days': True, 'reorder_cost': True, 'supplier_min_order_qty': True, 'supplier_order_multiple': True}

Quick stats:
                                 mean      min       max
stock_on_hand            81804.556922  16142.0  221817.0
lead_time_days               8.587416      5.0      14.0
reorder_cost                25.496205     20.0      31.0
supplier_min_order_qty      26.386461     12.0      72.0
supplier_order_multiple      7.848546      6.0      12.0


In [6]:
# Full pipeline smoke test
base = df.head(50000).copy()
cleaned = clean_pipeline(base)

must_have = [
    'stock_on_hand',
    'lead_time_days',
    'reorder_cost',
    'supplier_min_order_qty',
    'supplier_order_multiple',
]
missing_must_have = [c for c in must_have if c not in cleaned.columns]

assert len(missing_must_have) == 0, f'Missing expected columns: {missing_must_have}'
assert cleaned[must_have].isna().sum().sum() == 0, 'Synthetic fields contain NaNs'

print('Pipeline OK')
print('Input shape :', base.shape)
print('Output shape:', cleaned.shape)
print('Output dtypes sample:')
print(cleaned.dtypes.head(12))

AssertionError: Missing expected columns: ['stock_on_hand', 'lead_time_days', 'reorder_cost', 'supplier_min_order_qty', 'supplier_order_multiple']